##1.Partitioned Semantic Nets (Belief vs Reality)

In [ ]:
import itertools
from dataclasses import dataclass
from typing import Dict, List, Tuple
print("--- 1. Partitioned Semantic Nets ---")
partitions = {
    "Belief": [("Bird", "can_fly", True)],
    "Reality": [("Penguin", "is_a", "Bird"), ("Penguin", "can_fly", False)]
}

for p, facts in partitions.items():
    print(f"\nPartition {p}:")
    for f in facts:
        print(" ", f)

--- 1. Partitioned Semantic Nets ---

Partition Belief:
  ('Bird', 'can_fly', True)

Partition Reality:
  ('Penguin', 'is_a', 'Bird')
  ('Penguin', 'can_fly', False)


##2.Frames (Slot Inheritance)

In [ ]:
Animal = {"legs":4, "voice":"generic"}
Dog = {"is_a":Animal, "voice":"Bark"}
Labrador = {"is_a":Dog}


def get_slot(frame, slot):
    if slot in frame: return frame[slot]
    if "is_a" in frame: return get_slot(frame["is_a"], slot)
    return None


print("Labrador legs:", get_slot(Labrador,"legs"))
print("Labrador voice:", get_slot(Labrador,"voice"))


Labrador legs: 4
Labrador voice: Bark


 ## 3.Fuzzy Logic (Membership + Rule)

In [ ]:
print("\n--- 3. Fuzzy Logic ---")
def fuzzy_temp(x):
    return {
        "cold": max(0.0, min(1.0, (20 - x) / 10)),
        "warm": max(0.0, min(1.0, (x - 15) / 10, (30 - x) / 10)),
        "hot": max(0.0, min(1.0, (x - 25) / 10))
    }

t = 32
m = fuzzy_temp(t)
print("Temp=", t, "→", m)
print("Rule: IF hot THEN fan_high strength:", m["hot"])


--- 3. Fuzzy Logic ---
Temp= 32 → {'cold': 0.0, 'warm': 0.0, 'hot': 0.7}
Rule: IF hot THEN fan_high strength: 0.7


## 4.Hybrid (Neural + Symbolic Rule)

In [ ]:
print("\n--- 4. Hybrid ---")
def perceptron_temp(temp, th=25):
    return temp > th

temp = 27
hot = perceptron_temp(temp)
action = "Fan ON" if hot else "Fan OFF"
print(f"Temp {temp}°C → hot? {hot} → {action}")


--- 4. Hybrid ---
Temp 27°C → hot? True → Fan ON


## 5.Ontology (Tiny)

In [ ]:
print("\n--- 5. Ontology ---")
ontology = {"Alice": {"class": "Student", "university": "X University", "major": "CS"}}
print(ontology["Alice"])


--- 5. Ontology ---
{'class': 'Student', 'university': 'X University', 'major': 'CS'}


## 6.Scripts (Restaurant Visit)


In [ ]:
print("\n--- 6. Scripts ---")
script = [
    "Enter restaurant", "Wait to be seated", "Read menu & order",
    "Eat food", "Pay bill", "Leave"
]
for i, step in enumerate(script, 1):
    print(f"Step {i}: {step}")


--- 6. Scripts ---
Step 1: Enter restaurant
Step 2: Wait to be seated
Step 3: Read menu & order
Step 4: Eat food
Step 5: Pay bill
Step 6: Leave


##7. Expert System in AI (Diagnosis)

In [1]:
def expert_system(symptom):
    if symptom == "fever":
        return "Possible diagnosis: Flu"
    elif symptom == "chest pain":
        return "Possible diagnosis: Heart issue"
    else:
        return "Diagnosis unknown"


print(expert_system("fever"))


Possible diagnosis: Flu


## 8.Propositional Logic (Constraints / Brute-force SAT)

In [ ]:
print("\n--- 8. Propositional Logic ---")
clauses = [['A', 'B'], ['~A', 'C'], ['~B', '~C']]
vars_ = ['A', 'B', 'C']

def sat_solutions(vars_, clauses):
    sols = []
    for values in itertools.product([False, True], repeat=len(vars_)):
        assignment = dict(zip(vars_, values))
        ok = True
        for clause in clauses:
            if not any((lit.startswith("~") and not assignment[lit[1:]]) or
                       (not lit.startswith("~") and assignment[lit]) for lit in clause):
                ok = False
                break
        if ok:
            sols.append(assignment)
    return sols

print(sat_solutions(vars_, clauses))


--- 8. Propositional Logic ---
[{'A': False, 'B': True, 'C': False}, {'A': True, 'B': False, 'C': True}]


## 9.First-Order Logic (Horn Clauses)

In [ ]:
print("\n--- 9. First-Order Logic ---")
facts = {('Parent', ('Alice', 'Bob')), ('Parent', ('Bob', 'Charlie')), ('Human', ('Socrates',))}
rules = [
    (('Mortal', ('?x',)), [('Human', ('?x',))]),
    (('Grandparent', ('?x', '?z')), [('Parent', ('?x', '?y')), ('Parent', ('?y', '?z'))])
]

derived = set(facts)
changed = True
while changed:
    changed = False
    for head, body in rules:
        matches = []
        def match(body_idx, subst):
            if body_idx == len(body):
                matches.append(subst.copy())
                return
            pred, args = body[body_idx]
            for f_pred, f_args in derived:
                if f_pred == pred and len(f_args) == len(args):
                    new_subst = subst.copy()
                    ok = True
                    for va, vb in zip(args, f_args):
                        if va.startswith('?'):
                            if va in new_subst and new_subst[va] != vb:
                                ok = False
                                break
                            new_subst[va] = vb
                        elif va != vb:
                            ok = False
                            break
                    if ok:
                        match(body_idx + 1, new_subst)
        match(0, {})
        for subst in matches:
            h_pred, h_args = head
            newfact = (h_pred, tuple(subst.get(a, a) for a in h_args))
            if newfact not in derived:
                derived.add(newfact)
                changed = True

print("Derived facts:", derived)


--- 9. First-Order Logic ---
Derived facts: {('Parent', ('Bob', 'Charlie')), ('Parent', ('Alice', 'Bob')), ('Grandparent', ('Alice', 'Charlie')), ('Human', ('Socrates',)), ('Mortal', ('Socrates',))}


## 10. Disease Prediction (MYCIN)


In [ ]:
"""
mycin_disease_prediction.py
============================


A disease-prediction expert system modeled on MYCIN, the pioneering
1970s rule-based medical expert system built at Stanford.


What it borrows from MYCIN
---------------------------
1. Knowledge is stored as IF <symptom evidence> THEN <disease> rules,
   each carrying a Certainty Factor (CF) in the range [-1, 1] that
   expresses how strongly the rule's author trusts the conclusion
   when the evidence holds.
2. Evidence from the user is also given a CF, gathered with MYCIN's
   original 7-point verbal scale (definitely yes ... definitely no)
   instead of a raw probability, because clinicians reason more
   naturally in those terms.
3. Certainty factors combine using MYCIN's original formulas rather
   than plain probability theory:
     - AND across a rule's conditions -> MIN of the condition CFs
     - Rule firing -> CF(conclusion) = CF(rule) * antecedent_CF
     - Multiple rules concluding the same disease -> CF's combine
       with the MYCIN "evidence accumulation" formula (see
       combine_cf below), not simple addition or averaging.
4. Only conclusions whose combined CF exceeds a threshold (MYCIN
   used 0.2) are reported, and results are ranked by CF, mirroring
   MYCIN's differential-diagnosis style output.


This is a compact educational re-implementation, not the full
historical MYCIN (which had ~500 rules and a LISP-based inference
engine). The knowledge base below covers eight common illnesses and
can easily be extended.


Run it directly for an interactive Q&A diagnosis, or import
`MycinInference` to drive it programmatically (see `demo()` at the
bottom for an example with pre-supplied answers).
"""


from dataclasses import dataclass, field
from typing import Dict, List, Tuple




# ---------------------------------------------------------------------------
# 1. Certainty factor arithmetic (the heart of MYCIN's uncertainty model)
# ---------------------------------------------------------------------------


def combine_cf(cf1: float, cf2: float) -> float:
    """
    Combine two certainty factors for the same hypothesis using
    MYCIN's original evidence-combination formula.


    - Both positive  -> cf1 + cf2 * (1 - cf1)         (reinforcing belief)
    - Both negative  -> cf1 + cf2 * (1 + cf1)         (reinforcing disbelief)
    - Mixed signs    -> (cf1 + cf2) / (1 - min(|cf1|, |cf2|))  (conflict)
    """
    if cf1 >= 0 and cf2 >= 0:
        return cf1 + cf2 * (1 - cf1)
    if cf1 <= 0 and cf2 <= 0:
        return cf1 + cf2 * (1 + cf1)
    denom = 1 - min(abs(cf1), abs(cf2))
    if denom == 0:
        return 0.0
    return (cf1 + cf2) / denom




# MYCIN's classic verbal certainty scale for eliciting evidence from a user.
CERTAINTY_SCALE: Dict[str, float] = {
    "1": 1.0,    # Definitely yes
    "2": 0.8,    # Almost certainly yes
    "3": 0.6,    # Probably yes
    "4": 0.4,    # Maybe yes
    "5": 0.0,    # Unknown / don't know
    "6": -0.4,   # Maybe not
    "7": -0.8,   # Probably not
    "8": -1.0,   # Definitely not
}


CERTAINTY_SCALE_HELP = """
How certain are you that this symptom is present?
  1 = Definitely yes        (CF = 1.0)
  2 = Almost certainly yes  (CF = 0.8)
  3 = Probably yes          (CF = 0.6)
  4 = Maybe yes             (CF = 0.4)
  5 = Unknown                (CF = 0.0)
  6 = Maybe not              (CF = -0.4)
  7 = Probably not           (CF = -0.8)
  8 = Definitely not         (CF = -1.0)
"""




# ---------------------------------------------------------------------------
# 2. Knowledge base: IF (symptoms) THEN disease [CF]
# ---------------------------------------------------------------------------


@dataclass
class Rule:
    rule_id: str
    disease: str
    conditions: List[str]   # all symptoms must be present (ANDed, via MIN)
    cf: float                # the rule author's confidence in the conclusion




KNOWLEDGE_BASE: List[Rule] = [
    # --- Common Cold ---
    Rule("R1", "Common Cold", ["runny_nose", "sneezing", "sore_throat"], 0.7),
    Rule("R2", "Common Cold", ["cough", "runny_nose"], 0.5),
    Rule("R3", "Common Cold", ["sore_throat", "mild_fever"], 0.4),


    # --- Influenza (Flu) ---
    Rule("R4", "Influenza", ["fever", "body_ache", "fatigue", "chills"], 0.8),
    Rule("R5", "Influenza", ["fever", "cough", "headache"], 0.6),
    Rule("R6", "Influenza", ["sore_throat", "body_ache", "fatigue"], 0.5),


    # --- COVID-19 ---
    Rule("R7", "COVID-19", ["fever", "cough", "loss_of_taste_smell"], 0.9),
    Rule("R8", "COVID-19", ["breathing_difficulty", "fatigue", "cough"], 0.7),
    Rule("R9", "COVID-19", ["fever", "sore_throat", "loss_of_taste_smell"], 0.75),


    # --- Malaria ---
    Rule("R10", "Malaria", ["fever", "chills", "sweating"], 0.8),
    Rule("R11", "Malaria", ["fever", "headache", "vomiting", "chills"], 0.7),
    Rule("R12", "Malaria", ["joint_pain", "chills", "fever"], 0.6),


    # --- Typhoid ---
    Rule("R13", "Typhoid", ["fever", "abdominal_pain", "fatigue"], 0.7),
    Rule("R14", "Typhoid", ["fever", "diarrhea", "headache"], 0.6),
    Rule("R15", "Typhoid", ["loss_of_appetite", "fever", "fatigue"], 0.5),


    # --- Dengue ---
    Rule("R16", "Dengue", ["fever", "joint_pain", "rash"], 0.8),
    Rule("R17", "Dengue", ["fever", "headache", "eye_pain"], 0.7),
    Rule("R18", "Dengue", ["fatigue", "rash", "joint_pain"], 0.5),


    # --- Chickenpox ---
    Rule("R19", "Chickenpox", ["rash", "itching", "mild_fever"], 0.85),
    Rule("R20", "Chickenpox", ["rash", "fatigue", "loss_of_appetite"], 0.5),


    # --- Measles ---
    Rule("R21", "Measles", ["rash", "fever", "red_eyes"], 0.85),
    Rule("R22", "Measles", ["cough", "runny_nose", "rash"], 0.6),
    Rule("R23", "Measles", ["fever", "swollen_lymph_nodes", "rash"], 0.6),
]




def all_symptoms_in_kb() -> List[str]:
    """Return the sorted, de-duplicated list of every symptom the
    knowledge base asks about."""
    symptoms = set()
    for rule in KNOWLEDGE_BASE:
        symptoms.update(rule.conditions)
    return sorted(symptoms)




def all_diseases_in_kb() -> List[str]:
    return sorted({rule.disease for rule in KNOWLEDGE_BASE})




# ---------------------------------------------------------------------------
# 3. Inference engine
# ---------------------------------------------------------------------------


class MycinInference:
    """
    A small forward-chaining certainty-factor inference engine.


    Usage:
        engine = MycinInference()
        engine.set_symptom_cf("fever", 0.8)
        engine.set_symptom_cf("cough", 0.6)
        results = engine.diagnose()
    """


    CF_THRESHOLD = 0.2   # MYCIN's own reporting threshold


    def __init__(self, knowledge_base: List[Rule] = None):
        self.knowledge_base = knowledge_base or KNOWLEDGE_BASE
        self.symptom_cf: Dict[str, float] = {}


    def set_symptom_cf(self, symptom: str, cf: float) -> None:
        self.symptom_cf[symptom] = cf


    def _antecedent_cf(self, conditions: List[str]) -> float:
        """CF of a rule's IF-part = MIN of the CFs of its ANDed
        conditions. Missing evidence is treated as CF 0 (unknown)."""
        cfs = [self.symptom_cf.get(cond, 0.0) for cond in conditions]
        return min(cfs) if cfs else 0.0


    def diagnose(self) -> List[Tuple[str, float, List[str]]]:
        """
        Fire every rule, combine contributions per disease, and
        return (disease, combined_cf, contributing_rule_ids) sorted
        by combined_cf descending, filtered to CF > CF_THRESHOLD.
        """
        disease_cf: Dict[str, float] = {}
        disease_rules: Dict[str, List[str]] = {}


        for rule in self.knowledge_base:
            antecedent_cf = self._antecedent_cf(rule.conditions)
            if antecedent_cf <= 0:
                continue  # evidence doesn't support this rule firing at all


            contribution = rule.cf * antecedent_cf
            if rule.disease not in disease_cf:
                disease_cf[rule.disease] = contribution
                disease_rules[rule.disease] = [rule.rule_id]
            else:
                disease_cf[rule.disease] = combine_cf(
                    disease_cf[rule.disease], contribution
                )
                disease_rules[rule.disease].append(rule.rule_id)


        results = [
            (disease, cf, disease_rules[disease])
            for disease, cf in disease_cf.items()
            if cf > self.CF_THRESHOLD
        ]
        results.sort(key=lambda item: item[1], reverse=True)
        return results




# ---------------------------------------------------------------------------
# 4. Interactive CLI, MYCIN-style Q&A session
# ---------------------------------------------------------------------------


def ask_certainty(prompt: str) -> float:
    """Ask the user a symptom question and translate their answer
    (via CERTAINTY_SCALE, or a free-form y/n) into a CF value."""
    while True:
        answer = input(prompt).strip().lower()
        if answer in CERTAINTY_SCALE:
            return CERTAINTY_SCALE[answer]
        if answer in ("y", "yes"):
            return 1.0
        if answer in ("n", "no"):
            return -1.0
        if answer == "?":
            print(CERTAINTY_SCALE_HELP)
            continue
        print("Please answer 1-8 (type '?' to see the scale), or y/n.")




def run_interactive_session() -> None:
    print("=" * 70)
    print(" MYCIN-STYLE DISEASE PREDICTION EXPERT SYSTEM")
    print("=" * 70)
    print(
        "\nAnswer each question about your symptoms. You can reply with\n"
        "y/n, or type '?' once to see the full certainty scale (1-8) for\n"
        "finer-grained answers, MYCIN-style.\n"
    )


    engine = MycinInference()
    symptoms = all_symptoms_in_kb()


    for symptom in symptoms:
        readable = symptom.replace("_", " ")
        cf = ask_certainty(f"Do you have {readable}? (y/n or 1-8, '?' for help): ")
        if cf != 0.0:
            engine.set_symptom_cf(symptom, cf)


    print("\nAnalyzing symptoms against the rule base...\n")
    results = engine.diagnose()


    if not results:
        print(
            "No disease in the knowledge base was supported strongly enough\n"
            "(CF must exceed 0.2). This is not a medical diagnosis; please\n"
            "consult a doctor if you are unwell."
        )
        return


    print("Possible conditions, ranked by certainty factor:\n")
    for disease, cf, rule_ids in results:
        confidence_pct = round(cf * 100)
        print(f"  {disease:<15} CF = {cf:+.2f}  (~{confidence_pct}% confidence)"
              f"   [fired: {', '.join(rule_ids)}]")


    print(
        "\nNote: This is a simplified educational expert system inspired by\n"
        "MYCIN. It is NOT a substitute for professional medical advice."
    )




# ---------------------------------------------------------------------------
# 5. Non-interactive demo (useful for testing without stdin input)
# ---------------------------------------------------------------------------


def demo() -> None:
    """Runs a canned example so the engine's behavior can be inspected
    without answering interactive prompts."""
    print("Demo run: symptoms suggestive of Influenza / COVID-19\n")


    engine = MycinInference()
    sample_evidence = {
        "fever": 0.8,
        "cough": 0.6,
        "body_ache": 0.7,
        "fatigue": 0.6,
        "chills": 0.4,
        "loss_of_taste_smell": 0.6,
    }
    for symptom, cf in sample_evidence.items():
        engine.set_symptom_cf(symptom, cf)


    for disease, cf, rule_ids in engine.diagnose():
        print(f"  {disease:<15} CF = {cf:+.2f}   [fired: {', '.join(rule_ids)}]")




if __name__ == "__main__":
    import sys


    if len(sys.argv) > 1 and sys.argv[1] == "--demo":
        demo()
    else:
        run_interactive_session()

 MYCIN-STYLE DISEASE PREDICTION EXPERT SYSTEM

Answer each question about your symptoms. You can reply with
y/n, or type '?' once to see the full certainty scale (1-8) for
finer-grained answers, MYCIN-style.

